In [0]:
employee_df = spark.read.csv("/Volumes/quickstart_catalog/quickstart_schema/sandbox/dataset/employee.csv",header=True,inferSchema=True, sep='|',quote="'")

In [0]:
display(employee_df)

In [0]:
from pyspark.sql.functions import split, col

result_df = (
    employee_df.withColumn("skills", split(col("col_skills"), ","))
    .withColumn("current_salary", split(col("col_current_expected_salary"), ","))
    .drop("col_skills", "col_current_expected_salary")
)

display(result_df)

In [0]:
result_df.printSchema()

###Convert array of string to array of int

In [0]:
from pyspark.sql.functions import split, col

result_df = (
    employee_df.withColumn("skills", split(col("col_skills"), ","))
    .withColumn("current_salary", split(col("col_current_expected_salary"), ",").cast("array<int>"))
    .drop("col_skills", "col_current_expected_salary")
)

display(result_df)

In [0]:
result_df.printSchema()

###Accessing an array

In [0]:
result_df.select(
    col("current_salary"),
    col("current_salary")[0].alias("actual_salary"),
    col("current_salary").getItem(1).alias("expected_salary"),
).display()

In [0]:
result_df.withColumn(
    "actual_salary", col("current_salary").getItem(0)
).withColumn("expected_salary", col("current_salary").getItem(1)).display()

In [0]:
result_df.filter(col("current_salary").getItem(0) > col("current_salary").getItem(1)).select(col("name")).display()

In [0]:
from pyspark.sql.functions import size, array_distinct, array_contains

result_df.filter(array_contains("skills", "PySpark")).select(col("name")).display()

In [0]:

from pyspark.sql.functions import col, size, array_distinct, array_contains

result_df.select(
    col("name"),
    col("skills"),
    size(col("skills")).alias("skills_size"),
    array_distinct(col("skills")).alias("unique_skills"),
    array_contains(col("skills"), "PySpark").alias("has_pyspark")
).display()


In [0]:
result_df.filter((array_contains("skills", "PySpark")) & (array_contains("skills", "Hadoop")) & (size(col("skills")) >3)).select(col("name")).display()

In [0]:
df = spark.read.json("/Volumes/quickstart_catalog/quickstart_schema/sandbox/dataset/product_Information_001.json",multiLine=True)

In [0]:
df.printSchema()

In [0]:
df.select(col("details.features")).display()

In [0]:

df.filter(array_contains(col("details.features"), "Heart Rate Monitor")).select(col("name")).display()


In [0]:
from pyspark.sql.functions import explode
df.select(explode(col("details.features")).alias("features")).groupBy("features").count().display()

In [0]:

result_df.select(explode(col("skills")).alias("skill")).groupBy("skill").count().filter(col("count") >= 3).display()
